# Find-Enemy Training (Static Enemy, Single Police)

Goal: **train one police agent to find and go near a static enemy dot**, using your own drawn maps with walls.

Workflow:
- **Step 1 (local)**: Use the editor (e.g. `play_model.py`) to draw a map, place police (blue) and enemy (red), and save as `training_map.json`.
- **Step 2 (Colab or local notebook)**: Upload/copy `training_map.json` into this project folder and run the training cell; the env loads your map and trains the police.
- **Step 3**: Repeat with more advanced maps and continue training to improve the policy.

The enemy does **not move** in this notebook; only the police is trained to navigate toward the enemy.


In [ ]:
# If running in Colab: clone repo and install dependencies (run once per session)
import sys, os
if "google.colab" in sys.modules:
    %cd /content
    if not os.path.exists("pursuit_arena"):
        !git clone https://github.com/GraslyDias/pursuit_arena.git
    %cd /content/pursuit_arena
    !pip install pygame gymnasium stable-baselines3[extra] numpy


## Upload or provide `training_map.json`

If you're running in Colab, upload your locally saved `training_map.json` from the editor.


In [ ]:
import sys
if "google.colab" in sys.modules:
    from google.colab import files
    uploaded = files.upload()  # choose training_map.json from your computer


## Train POLICE to find static enemy on your map

This uses `ChaseEscapeEnv` with `static_enemy=True` and **always** loads `training_map.json` if present.


In [ ]:
from pathlib import Path

from stable_baselines3 import PPO
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.vec_env import DummyVecEnv

import pursuit_arena.ai.rl.chase_escape_env as ce
from pursuit_arena.ai.rl.chase_escape_env import ChaseEscapeEnv, load_training_map

# Reward config for FIND-ENEMY task (5-second budget, strong signal for finding enemy)
# You can tune these numbers.
ce.POLICE_REWARD_ARREST = 10.0        # big reward for reaching (arresting) the enemy
ce.POLICE_REWARD_ESCAPE = 0.0         # enemy never moves; escape is not focus here
ce.POLICE_REWARD_VISIBLE = 0.2        # reward per step when enemy is in FOV/LOS
ce.POLICE_PENALTY_WALL_COLLISION = -0.1
ce.POLICE_STEP_PENALTY = -0.001
ce.POLICE_TIMEOUT_PENALTY = -5.0      # strong negative when time is over and enemy not found

log_dir = Path("runs/ppo_find_enemy")
log_dir.mkdir(parents=True, exist_ok=True)

MAX_STEPS_PER_EPISODE = 300  # ~5 seconds if you imagine 60 FPS

def make_env():
    env = ChaseEscapeEnv(render_mode=None, max_steps=MAX_STEPS_PER_EPISODE)
    env.static_enemy = True  # enemy does not move
    map_path = Path("training_map.json")
    if map_path.exists():
        env.training_map_options = load_training_map(map_path)
        print("Using training_map.json for find-enemy training")
    env = Monitor(env, str(log_dir / "monitor_find_enemy.csv"))
    return env

vec_env = DummyVecEnv([make_env])

model_path = log_dir / "ppo_find_enemy_final.zip"
if model_path.exists():
    model = PPO.load(str(model_path), env=vec_env, device="cpu")
    print("Loaded existing find-enemy model, continuing training.")
else:
    model = PPO("MlpPolicy", vec_env,
                verbose=1,
                tensorboard_log=str(log_dir / "tb_find_enemy"),
                device="cpu")
    print("Training find-enemy model from scratch.")

TOTAL_TIMESTEPS = 200_000  # adjust as you like
model.learn(total_timesteps=TOTAL_TIMESTEPS)
model.save(str(model_path))
print("Saved find-enemy model to", model_path)
vec_env.close()

# --- Simple evaluation to show episode rewards and steps ---
env_eval = ChaseEscapeEnv(render_mode=None, max_steps=MAX_STEPS_PER_EPISODE)
env_eval.static_enemy = True
map_path = Path("training_map.json")
if map_path.exists():
    env_eval.training_map_options = load_training_map(map_path)

model_eval = PPO.load(str(model_path), env=env_eval, device="cpu")

num_episodes = 10
for ep in range(num_episodes):
    obs, info = env_eval.reset()
    done, truncated = False, False
    ep_reward = 0.0
    steps = 0
    while not (done or truncated):
        action, _ = model_eval.predict(obs, deterministic=True)
        obs, reward, done, truncated, info = env_eval.step(action)
        ep_reward += float(reward)
        steps += 1
    print(f"Episode {ep+1}: steps={steps}, reward={ep_reward:.3f}, info={info}")

env_eval.close()


## Quick visual check (optional, local Jupyter)

If you're running this notebook **locally** (not Colab), you can use `render_mode='human'` to open a window and watch the agent.


In [ ]:
from stable_baselines3 import PPO
from pursuit_arena.ai.rl.chase_escape_env import ChaseEscapeEnv, load_training_map

try:
    # Only works on local machine with a display (not in Colab)
    env = ChaseEscapeEnv(render_mode="human")
    env.static_enemy = True
    map_path = Path("training_map.json")
    if map_path.exists():
        env.training_map_options = load_training_map(map_path)
    model = PPO.load("runs/ppo_find_enemy/ppo_find_enemy_final.zip", env=env)

    episodes = 3
    for ep in range(episodes):
        obs, info = env.reset()
        done = False
        truncated = False
        while not (done or truncated):
            action, _ = model.predict(obs, deterministic=True)
            obs, reward, done, truncated, info = env.step(action)
    env.close()
except Exception as e:
    print("Visual run skipped or failed (likely due to no display):", e)
